# 19.19 Spurious correlations & shortcut learning

Shortcut learning occurs when a model uses a feature that predicts training labels but fails under shift.

ERM rewards any feature that lowers training loss, including accidental cues. This notebook measures the environment gap and then ablates the proxy cue to show whether the model learned the stable core signal.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1919
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def _standardize(X):
    scaler = StandardScaler()
    return scaler.fit_transform(np.asarray(X, dtype=float))


def _binary_target(y):
    y = np.asarray(y)
    classes = np.unique(y)
    if len(classes) == 2:
        return (y == classes[-1]).astype(int)
    return (y == classes[-1]).astype(int)


def clf_ladder():
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def fit_logistic(X, y):
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X, y)
    return clf


## Build the shortcut audit once on D1\n\nThe lesson formula is $$\Delta_{env}=R_{shift}(h)-R_{train}(h).$$\nUsing the lesson components, $0.100+0.320+0.220=0.640$ and the leading shortcut share is $|0.100|/0.640=0.156$.

In [ ]:

def error_rate(model, env):
    X, y = env
    pred = model.predict(X)
    return 1.0 - accuracy_score(y, pred)


def environment_gap(model, env_train, env_shift):
    train_risk = error_rate(model, env_train)
    shift_risk = error_rate(model, env_shift)
    return shift_risk - train_risk


def shortcut_ablation(model, X):
    X_keep_core = X.copy()
    X_keep_core[:, -1] = 0.0
    full_prob = model.predict_proba(X)[:, 1]
    ablated_prob = model.predict_proba(X_keep_core)[:, 1]
    return np.mean(np.abs(full_prob - ablated_prob))

components = np.array([0.100, 0.320, 0.220])
total = float(components.sum())
share = float(abs(components[0]) / np.abs(components).sum())
assert round(total, 3) == 0.640
assert round(share, 3) == 0.156
print("lesson total", round(total, 3))
print("leading shortcut share", round(share, 3))


Now make the D1 shortcut concrete: the core feature remains stable, while the cue feature flips in the shifted environment.

In [ ]:

def make_shortcut_environment(X, y, shortcut_strength, shift_strength, seed):
    local_rng = np.random.default_rng(seed)
    Xs = _standardize(X)
    yb = _binary_target(y)
    core = Xs[:, : min(3, Xs.shape[1])]
    cue_train = np.where(local_rng.random(len(yb)) < shortcut_strength, yb, 1 - yb)
    cue_shift = np.where(local_rng.random(len(yb)) < shift_strength, yb, 1 - yb)
    cue_train = (2 * cue_train - 1).reshape(-1, 1)
    cue_shift = (2 * cue_shift - 1).reshape(-1, 1)
    env_train = (np.column_stack([core, cue_train]), yb)
    env_shift = (np.column_stack([core, cue_shift]), yb)
    return env_train, env_shift

base_name, base_X, base_y = clf_ladder()[0]
env_train, env_shift = make_shortcut_environment(base_X, base_y, 1.0, 0.0, SEED)
model = fit_logistic(env_train[0], env_train[1])
gap = environment_gap(model, env_train, env_shift)
ablation = shortcut_ablation(model, env_shift[0])
print(base_name)
print("environment gap", round(gap, 3))
print("cue ablation impact", round(ablation, 3))


## Dataset ladder

We reuse the F15 classification ladder and add a shortcut cue with rising shift severity.

In [ ]:

shortcut_specs = [
    ("D1 linear toy", 1.00, 0.00),
    ("D2 interaction shortcut", 0.95, 0.25),
    ("D3 real tabular proxy", 0.90, 0.35),
    ("D4 digits-style stress", 0.85, 0.45),
    ("D5 shifted biased shortcut", 0.80, 0.55),
]
shortcut_rungs = []
for idx, ((base_name, X, y), (name, train_strength, shift_strength)) in enumerate(zip(clf_ladder(), shortcut_specs)):
    env_train, env_shift = make_shortcut_environment(X, y, train_strength, shift_strength, SEED + idx)
    shortcut_rungs.append((name, env_train, env_shift, train_strength, shift_strength))
    print(name, env_train[0].shape, "classes", np.bincount(env_train[1]))
    print("sample", np.round(env_train[0][:2], 3))


## Run the same audit across D1-D5

In [ ]:

shortcut_results = []
for name, env_train, env_shift, train_strength, shift_strength in shortcut_rungs:
    model = fit_logistic(env_train[0], env_train[1])
    gap = environment_gap(model, env_train, env_shift)
    robust_acc = 1.0 - error_rate(model, env_shift)
    cue_weight = abs(model.coef_[0, -1])
    core_weight = float(np.mean(np.abs(model.coef_[0, :-1])))
    ablation = shortcut_ablation(model, env_shift[0])
    shortcut_results.append((name, robust_acc, gap, core_weight, cue_weight, ablation))

print("rung | robust_acc | env_gap | core_w | cue_w | ablation")
for row in shortcut_results:
    print(row[0], *(round(float(v), 3) for v in row[1:]))


## Results visualization

Left: core-vs-cue attribution panels. Right: environment gap as shift stress rises.

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.ravel()
for ax, row in zip(axes[:5], shortcut_results):
    name, robust_acc, gap, core_weight, cue_weight, ablation = row
    ax.bar(["core", "cue"], [core_weight, cue_weight], color=["steelblue", "tomato"])
    ax.set_title(name)
    ax.set_ylim(0, max(core_weight, cue_weight, 0.1) * 1.25)
axes[5].plot(range(1, 6), [row[2] for row in shortcut_results], marker="o")
axes[5].set_xticks(range(1, 6))
axes[5].set_xlabel("rung")
axes[5].set_ylabel("environment gap")
axes[5].set_title("Gap vs shift")
plt.tight_layout()


## Pitfall on D5: validating on the same environment

The wrong audit evaluates train-like data twice. The fix uses an environment split and ablates the proxy cue.

In [ ]:

name, env_train, env_shift, train_strength, shift_strength = shortcut_rungs[-1]
model = fit_logistic(env_train[0], env_train[1])
wrong_gap = environment_gap(model, env_train, env_train)
fixed_gap = environment_gap(model, env_train, env_shift)
proxy_impact = shortcut_ablation(model, env_shift[0])
print("wrong same-environment gap", round(wrong_gap, 3))
print("fixed environment-split gap", round(fixed_gap, 3))
print("proxy ablation impact", round(proxy_impact, 3))


## Evaluate it + Practice

- Metric: environment gap and robust accuracy
- No-skill baseline: same-environment validation reports zero shift
- Cheap sanity check: flip the cue and ensure the gap changes
- Ablation: zero the cue feature and watch shortcut impact drop
- Failure signals: large cue coefficient, high gap, or conclusions that flip after ablation

Practice prompts:
1. Change the D5 stress level and predict which metric should move first.

2. Add one subgroup or environment slice and explain whether the conclusion changes.

3. Replace the default logistic model with another CPU-only sklearn classifier and compare the same metric.